# BattleNet Training on Kaggle GPU

**Dataset Structure:**
- Input (`/kaggle/input/`): Same as local `KIIT-MiTA/` - images in `train/images/`, YOLO labels in `train/labels/`
- Working (`/kaggle/working/`): Generated JSON annotations stored here

**Optimized hyperparameters:**
- Weight Decay: 1e-3 (+2.43% improvement)
- Batch Size: 32
- Augmentation: baseline (strong hurt performance)
- Loss: BCE (focal hurt -3.5%)

In [ ]:
import json
import sys
from pathlib import Path
from datetime import datetime
from glob import glob

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from PIL import Image
import numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Path Configuration

In [ ]:
# Kaggle paths
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# Find dataset directory (the one with train/valid/test folders)
dataset_dirs = list(INPUT_ROOT.glob("*"))
DATASET_ROOT = None

print("Scanning input directories...")
for d in dataset_dirs:
    print(f"  - {d.name}")
    # Check if this is the KIIT-MiTA dataset
    if (d / "train" / "images").exists():
        DATASET_ROOT = d
        print(f"    -> Found dataset structure!")
        break

if DATASET_ROOT is None:
    raise ValueError("Could not find KIIT-MiTA dataset in /kaggle/input/")

print(f"\nDataset root: {DATASET_ROOT}")
print(f"\nDataset structure:")
for split in ['train', 'valid', 'test']:
    images_dir = DATASET_ROOT / split / "images"
    labels_dir = DATASET_ROOT / split / "labels"
    num_images = len(list(images_dir.glob("*"))) if images_dir.exists() else 0
    num_labels = len(list(labels_dir.glob("*"))) if labels_dir.exists() else 0
    print(f"  {split}: {num_images} images, {num_labels} labels")

# Class names
CLASS_NAMES = ["Artilary", "Missile", "Radar", "M. Rocket Launcher", "Soldier", "Tank", "Vehicle"]
NUM_CLASSES = len(CLASS_NAMES)

## Convert YOLO Labels to JSON Format

YOLO format: `class_id center_x center_y width height` (normalized 0-1)

Convert to multi-label JSON format for training.

In [ ]:
def convert_yolo_to_json(split):
    """Convert YOLO TXT labels to JSON multi-label format."""
    images_dir = DATASET_ROOT / split / "images"
    labels_dir = DATASET_ROOT / split / "labels"
    
    annotations = []
    
    # Get all image files
    image_files = sorted(list(images_dir.glob("*")))
    
    for img_path in tqdm(image_files, desc=f"Converting {split}"):
        # Get corresponding label file
        label_path = labels_dir / (img_path.stem + ".txt")
        
        # Create multi-hot label vector
        labels = np.zeros(NUM_CLASSES)
        class_ids = []
        
        if label_path.exists():
            with open(label_path, 'r') as f:
                lines = f.readlines()
                
            for line in lines:
                line = line.strip()
                if line:
                    parts = line.split()
                    if len(parts) >= 1:
                        class_id = int(parts[0])
                        if 0 <= class_id < NUM_CLASSES:
                            labels[class_id] = 1
                            class_ids.append(class_id)
        
        # Create annotation entry
        rel_path = f"{split}/images/{img_path.name}"
        annotations.append({
            "image_path": rel_path,
            "label": labels.tolist(),
            "class_ids": sorted(class_ids)
        })
    
    return annotations

# Convert all splits and save to working directory
print("Converting YOLO labels to JSON format...")
for split in ['train', 'valid', 'test']:
    annotations = convert_yolo_to_json(split)
    output_path = WORK_DIR / f"{split}_annotations.json"
    
    with open(output_path, 'w') as f:
        json.dump(annotations, f, indent=2)
    
    print(f"  {split}: {len(annotations)} annotations saved to {output_path.name}")

print("\nAnnotation files created in /kaggle/working/")

## Training Configuration

In [ ]:
CONFIG = {
    "architecture": "battlenet",
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,

    # Batch / image
    "batch_size": 32,
    "image_size": 224,
    "num_workers": 4,

    # Training schedule
    "epochs_head": 15,       # Phase 1: frozen backbone, head only
    "epochs_finetune": 25,   # Phase 2: full fine-tune with differential LRs
    "lr_head": 1e-3,
    "lr_finetune": 1e-4,

    # Optimization — Adam + weight_decay proven best across 42 experiments
    "optimizer": "adam",
    "weight_decay": 1e-3,
    "clip_grad_norm": 1.0,

    # Augmentation / loss
    "augmentation": "baseline",
    "loss_type": "bce",
    "pos_weight": 1.0,

    # Architecture — BattleNet: Swin-T backbone + military-specialized BattleNetHead
    #   Swin-T (768d, 28M): best single baseline at 0.8326 Micro F1
    #   BattleNetHead adds: SE recalibration, residual MLP, LabelCoOccurrence,
    #   and auxiliary supervision — all absent in the vanilla linear head baseline
    "head_dim": 768,
    "dropout": 0.3,
    "aux_loss_weight": 0.3,  # weight for auxiliary head loss during training

    # Inference — TTA + per-class threshold optimization on val set
    "tta": True,
    "optimize_thresholds": True,

    # Device
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "mixed_precision": torch.cuda.is_available(),
}

print("\n=== BattleNet Configuration ===")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Dataset and Dataloaders

In [ ]:
class KiitMitaDataset(torch.utils.data.Dataset):
    """KIIT-MiTA multi-label classification dataset."""
    
    def __init__(self, annotations_path, dataset_root, transform=None):
        with open(annotations_path, 'r') as f:
            self.annotations = json.load(f)
        self.dataset_root = Path(dataset_root)
        self.transform = transform
        self.num_classes = NUM_CLASSES
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_path = self.dataset_root / ann["image_path"]
        image = Image.open(img_path).convert('RGB')
        
        # Labels already in multi-hot format
        labels = torch.tensor(ann["label"], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        
        return image, labels

# Data transforms
def get_transforms(augmentation="baseline", image_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    return train_transform, val_transform

# Load datasets
train_ann_path = WORK_DIR / "train_annotations.json"
val_ann_path = WORK_DIR / "valid_annotations.json"
test_ann_path = WORK_DIR / "test_annotations.json"

train_transform, val_transform = get_transforms(CONFIG["augmentation"], CONFIG["image_size"])

train_dataset = KiitMitaDataset(train_ann_path, DATASET_ROOT, train_transform)
val_dataset = KiitMitaDataset(val_ann_path, DATASET_ROOT, val_transform)
test_dataset = KiitMitaDataset(test_ann_path, DATASET_ROOT, val_transform)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])
val_loader = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
test_loader = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Verify label distribution
print("\nLabel distribution in training set:")
label_counts = np.zeros(NUM_CLASSES)
for i in range(len(train_dataset)):
    _, labels = train_dataset[i]
    label_counts += labels.numpy()

for i, count in enumerate(label_counts):
    print(f"  {CLASS_NAMES[i]:<20} {int(count)} samples")

## BattleNet Model Architecture

In [ ]:
import torch
import torch.nn as nn


class SqueezeExcitation(nn.Module):
    """SE channel recalibration — adaptively reweights feature channels.

    Learns which feature dimensions are most discriminative for military
    object classification (e.g. texture channels for camouflage detection).
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.fc(x)


class LabelCoOccurrence(nn.Module):
    """Learnable label co-occurrence module.

    Captures military object relationships (e.g. Tank+Soldier co-deployment,
    Radar+Missile battery groupings) via a learnable interaction matrix W.
    The learnable scale parameter controls how strongly co-occurrence
    patterns influence the final logits.
    """
    def __init__(self, num_classes):
        super().__init__()
        self.W = nn.Parameter(torch.eye(num_classes) + torch.randn(num_classes, num_classes) * 0.01)
        self.bias = nn.Parameter(torch.zeros(num_classes))
        self.scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, logits):
        scale = torch.clamp(self.scale, 0.0, 1.0)
        return logits + torch.matmul(logits, self.W.t()) * scale + self.bias


class BattleNetHead(nn.Module):
    """Military-specialized classification head.

    Novel contributions over the vanilla Swin-T linear head:
      1. SE recalibration  — channel-wise feature reweighting (residual)
      2. Residual MLP      — deeper non-linear transformation with skip connection
      3. LabelCoOccurrence — explicit military object co-occurrence modelling
      4. Auxiliary head    — secondary supervision for richer gradient flow

    Baseline (vanilla Swin-T): Linear(768, 7) → 0.8326 Micro F1
    BattleNet (this head):     all of the above on the same 768d features
    """
    def __init__(self, feat_dim, num_classes=7, head_dim=768, dropout=0.3):
        super().__init__()

        # Project backbone features into head space
        self.proj = nn.Sequential(
            nn.Linear(feat_dim, head_dim),
            nn.LayerNorm(head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # SE channel recalibration (residual + BN for training stability)
        self.se = SqueezeExcitation(head_dim, reduction=16)
        self.se_bn = nn.BatchNorm1d(head_dim)

        # Residual MLP block
        self.mlp = nn.Sequential(
            nn.Linear(head_dim, head_dim),
            nn.LayerNorm(head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Primary classifier + label co-occurrence refinement
        self.classifier = nn.Linear(head_dim, num_classes)
        self.label_cooccur = LabelCoOccurrence(num_classes)

        # Auxiliary head — provides additional gradient signal during training,
        # discarded at inference
        self.aux_head = nn.Sequential(
            nn.Linear(head_dim, head_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_dim // 2, num_classes),
        )

    def forward(self, x, return_aux=False):
        x = self.proj(x)
        x = x + self.se_bn(self.se(x))   # residual SE
        x = x + self.mlp(x)              # residual MLP
        logits = self.label_cooccur(self.classifier(x))
        if return_aux:
            return logits, self.aux_head(x)
        return logits


class BattleNet(nn.Module):
    """BattleNet: Swin-T backbone with military-specialized BattleNetHead.

    Architecture motivation:
      The vanilla Swin-T achieves 0.8326 Micro F1 with a single Linear(768,7) head.
      BattleNet replaces that head with BattleNetHead, which exploits:
        - channel-level feature importance (SE)
        - deeper non-linear transformations (residual MLP)
        - explicit label dependency modelling (LabelCoOccurrence)
        - richer gradient flow via auxiliary supervision

    Training strategy:
      Phase 1 (15 epochs): freeze backbone, train BattleNetHead only
      Phase 2 (25 epochs): unfreeze all with differential LRs
                           backbone @ lr/10 (1e-5), head @ lr (1e-4)
    """
    def __init__(self, num_classes=7, pretrained=True, head_dim=768, dropout=0.3):
        super().__init__()

        from torchvision.models import swin_t, Swin_T_Weights
        self.backbone = swin_t(
            weights=Swin_T_Weights.IMAGENET1K_V1 if pretrained else None
        )
        feat_dim = self.backbone.head.in_features  # 768
        self.backbone.head = nn.Identity()         # remove original linear head

        self.head = BattleNetHead(feat_dim, num_classes, head_dim, dropout)

    def forward(self, x, return_aux=False):
        return self.head(self.backbone(x), return_aux=return_aux)


# Create model
model = BattleNet(
    num_classes=CONFIG["num_classes"],
    pretrained=True,
    head_dim=CONFIG["head_dim"],
    dropout=CONFIG["dropout"],
).to(CONFIG["device"])

backbone_params = sum(p.numel() for p in model.backbone.parameters())
head_params = sum(p.numel() for p in model.head.parameters())
total_params = backbone_params + head_params

print(f"Backbone (Swin-T):  {backbone_params:>12,} params")
print(f"BattleNetHead:      {head_params:>12,} params")
print(f"Total:              {total_params:>12,} params")
print(f"Feature dim:        {model.backbone.head.in_features if hasattr(model.backbone.head, 'in_features') else 768}d  (Swin-T native)")
print(f"\nBaseline to beat:   Swin-T linear head  →  0.8326 Micro F1")

## Training Setup

In [ ]:
# Loss function
criterion = nn.BCEWithLogitsLoss()

# Metrics computation
def compute_metrics(probs, labels):
    preds = (probs > 0.5).astype(np.float32)
    metrics = {}
    for i, class_name in enumerate(CONFIG["class_names"]):
        tp = np.sum((preds[:, i] == 1) & (labels[:, i] == 1))
        fp = np.sum((preds[:, i] == 1) & (labels[:, i] == 0))
        fn = np.sum((preds[:, i] == 0) & (labels[:, i] == 1))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        metrics[class_name] = {"precision": precision, "recall": recall, "f1": f1}
    total_tp = np.sum((preds == 1) & (labels == 1))
    total_fp = np.sum((preds == 1) & (labels == 0))
    total_fn = np.sum((preds == 0) & (labels == 1))
    metrics["micro"] = {
        "precision": total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0,
        "recall": total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0,
        "f1": 2 * total_tp / (2 * total_tp + total_fp + total_fn) if (2 * total_tp + total_fp + total_fn) > 0 else 0.0,
    }
    metrics["exact_match"] = np.all(preds == labels, axis=1).mean()
    return metrics

# Training history
history = {"train_loss": [], "val_loss": [], "val_f1": []}

print("Training setup complete.")
print(f"  Criterion: BCEWithLogitsLoss")
print(f"  Aux loss weight: {CONFIG['aux_loss_weight']} (auxiliary head, training only)")

## Training Loop

### Phase 1: Train Head Only (Backbone Frozen)

In [ ]:
device = torch.device(CONFIG["device"])
scaler = torch.cuda.amp.GradScaler() if CONFIG["mixed_precision"] else None

# Phase 1: freeze backbone, train BattleNetHead only
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer = optim.Adam(
    model.head.parameters(),
    lr=CONFIG["lr_head"],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["epochs_head"], eta_min=1e-6
)

best_val_f1 = 0.0

print(f"\n=== Phase 1: Training BattleNetHead (Backbone Frozen) — {CONFIG['epochs_head']} epochs ===")
print(f"Trainable params: {sum(p.numel() for p in model.head.parameters()):,}")

for epoch in range(CONFIG["epochs_head"]):
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs_head']}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                logits, aux_logits = model(images, return_aux=True)
                w = CONFIG["aux_loss_weight"]
                loss = (1 - w) * criterion(logits, labels) + w * criterion(aux_logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.head.parameters(), CONFIG["clip_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, aux_logits = model(images, return_aux=True)
            w = CONFIG["aux_loss_weight"]
            loss = (1 - w) * criterion(logits, labels) + w * criterion(aux_logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.head.parameters(), CONFIG["clip_grad_norm"])
            optimizer.step()

        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= len(train_loader.dataset)
    scheduler.step()

    # Validate
    model.eval()
    val_loss = 0.0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            val_loss += criterion(logits, labels).item() * images.size(0)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    metrics = compute_metrics(all_probs, all_labels)
    val_f1 = metrics["micro"]["f1"]

    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["val_f1"].append(float(val_f1))

    print(f"  Epoch {epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "f1": best_val_f1, "config": CONFIG},
                   WORK_DIR / "battlenet_head_best.pth")

print(f"\nPhase 1 complete. Best Val F1: {best_val_f1:.4f}")

### Phase 2: Fine-tune Entire Network

In [ ]:
# Phase 2: unfreeze all, fine-tune with differential learning rates
for param in model.parameters():
    param.requires_grad = True

# Backbone gets 10x lower LR to protect pretrained Swin-T weights
optimizer = optim.Adam(
    [
        {"params": list(model.backbone.parameters()), "lr": CONFIG["lr_finetune"] / 10},  # 1e-5
        {"params": list(model.head.parameters()),     "lr": CONFIG["lr_finetune"]},        # 1e-4
    ],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG["epochs_finetune"], eta_min=1e-7
)

best_val_f1 = 0.0
start_epoch = CONFIG["epochs_head"]

print(f"\n=== Phase 2: Fine-tuning Entire Network — {CONFIG['epochs_finetune']} epochs ===")
print(f"Backbone LR: {CONFIG['lr_finetune'] / 10:.1e} | Head LR: {CONFIG['lr_finetune']:.1e}")

for epoch in range(CONFIG["epochs_finetune"]):
    model.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {start_epoch+epoch+1}/{start_epoch+CONFIG['epochs_finetune']}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                logits, aux_logits = model(images, return_aux=True)
                w = CONFIG["aux_loss_weight"]
                loss = (1 - w) * criterion(logits, labels) + w * criterion(aux_logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["clip_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, aux_logits = model(images, return_aux=True)
            w = CONFIG["aux_loss_weight"]
            loss = (1 - w) * criterion(logits, labels) + w * criterion(aux_logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["clip_grad_norm"])
            optimizer.step()

        train_loss += loss.item() * images.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    train_loss /= len(train_loader.dataset)
    scheduler.step()

    # Validate
    model.eval()
    val_loss = 0.0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            val_loss += criterion(logits, labels).item() * images.size(0)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    val_loss /= len(val_loader.dataset)
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    metrics = compute_metrics(all_probs, all_labels)
    val_f1 = metrics["micro"]["f1"]

    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["val_f1"].append(float(val_f1))

    print(f"  Epoch {start_epoch+epoch+1:2d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | LR: {optimizer.param_groups[1]['lr']:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({"epoch": start_epoch + epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "f1": best_val_f1, "config": CONFIG},
                   WORK_DIR / "battlenet_finetune_best.pth")

print(f"\nPhase 2 complete. Best Val F1: {best_val_f1:.4f}")

## Test Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(WORK_DIR / "battlenet_finetune_best.pth", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

def get_probs(model, dataloader, use_tta=False):
    """Collect sigmoid probabilities from a dataloader, optionally with h-flip TTA."""
    model.eval()
    total_loss = 0.0
    all_probs, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Inference", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)

            if use_tta:
                logits_flip = model(torch.flip(images, dims=[-1]))
                probs = (probs + torch.sigmoid(logits_flip)) / 2.0

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)
    return avg_loss, np.concatenate(all_probs), np.concatenate(all_labels)

def compute_metrics_with_thresholds(probs, labels, thresholds=None):
    """Compute metrics using per-class thresholds (default 0.5 for all)."""
    if thresholds is None:
        thresholds = np.full(probs.shape[1], 0.5)
    preds = (probs > thresholds[np.newaxis, :]).astype(np.float32)
    metrics = {}
    for i, class_name in enumerate(CONFIG["class_names"]):
        tp = np.sum((preds[:, i] == 1) & (labels[:, i] == 1))
        fp = np.sum((preds[:, i] == 1) & (labels[:, i] == 0))
        fn = np.sum((preds[:, i] == 0) & (labels[:, i] == 1))
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        metrics[class_name] = {"precision": float(precision), "recall": float(recall), "f1": float(f1)}
    total_tp = np.sum((preds == 1) & (labels == 1))
    total_fp = np.sum((preds == 1) & (labels == 0))
    total_fn = np.sum((preds == 0) & (labels == 1))
    metrics["micro"] = {
        "precision": float(total_tp / (total_tp + total_fp)) if (total_tp + total_fp) > 0 else 0.0,
        "recall": float(total_tp / (total_tp + total_fn)) if (total_tp + total_fn) > 0 else 0.0,
        "f1": float(2 * total_tp / (2 * total_tp + total_fp + total_fn)) if (2 * total_tp + total_fp + total_fn) > 0 else 0.0,
    }
    metrics["exact_match"] = float(np.all(preds == labels, axis=1).mean())
    return metrics

use_tta = CONFIG.get("tta", False)

# Step 1: Get val probs to find per-class optimal thresholds
_, val_probs, val_labels = get_probs(model, val_loader, use_tta=use_tta)

thresholds = np.full(CONFIG["num_classes"], 0.5)
if CONFIG.get("optimize_thresholds", False):
    print("Optimizing per-class thresholds on validation set...")
    for i, class_name in enumerate(CONFIG["class_names"]):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.2, 0.81, 0.05):
            preds_i = (val_probs[:, i] > t).astype(np.float32)
            tp = np.sum((preds_i == 1) & (val_labels[:, i] == 1))
            fp = np.sum((preds_i == 1) & (val_labels[:, i] == 0))
            fn = np.sum((preds_i == 0) & (val_labels[:, i] == 1))
            f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[i] = best_t
        print(f"  {class_name:<22} threshold={best_t:.2f}  val_f1={best_f1:.4f}")

# Step 2: Evaluate on test set with optimized thresholds + TTA
test_loss, test_probs, test_labels = get_probs(model, test_loader, use_tta=use_tta)
test_metrics = compute_metrics_with_thresholds(test_probs, test_labels, thresholds)

print(f"\n=== Test Results {'(TTA + optimized thresholds)' if use_tta else ''} ===")
print(f"Test Loss:             {test_loss:.4f}")
print(f"Exact Match Accuracy:  {test_metrics['exact_match']:.4f}")
print(f"Micro F1:              {test_metrics['micro']['f1']:.4f}")
print(f"Micro Precision:       {test_metrics['micro']['precision']:.4f}")
print(f"Micro Recall:          {test_metrics['micro']['recall']:.4f}")
print(f"Thresholds used:       {dict(zip(CONFIG['class_names'], thresholds.round(2)))}")
print("\nPer-Class F1 Scores:")
for class_name in CONFIG["class_names"]:
    print(f"  {class_name:<22} {test_metrics[class_name]['f1']:.4f}")

## Save Results

In [ ]:
# Save final results
results = {
    "config": CONFIG,
    "history": history,
    "test_metrics": {k: float(v) if isinstance(v, (np.float64, np.float32)) else v 
                     for k, v in test_metrics.items()},
    "test_loss": float(test_loss),
    "best_val_f1": float(best_val_f1),
}

with open(WORK_DIR / "battlenet_results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to {WORK_DIR / 'battlenet_results.json'}")
print(f"Best checkpoint: {WORK_DIR / 'battlenet_finetune_best.pth'}")
print(f"\nFiles in /kaggle/working/:")
for f in WORK_DIR.glob("*"):
    size_mb = f.stat().st_size / 1024 / 1024 if f.is_file() else 0
    print(f"  {f.name:<30} {size_mb:>8.2f} MB")